# 01 · Baseline (A0), external test, demo

ImageNet → APTOS. Cleans APTOS (submission template ignored, duplicates removed), trains A0, evaluates on APTOS test and on IDRiD, makes the README figures, saves everything under the tag `mvp`, and deploys the in-browser demo.

**Kaggle settings:** Accelerator *GPU T4 x2* (the code uses one GPU), Internet *on*,
secret `HF_TOKEN` (a Hugging Face token with write access) switched on for this notebook.
**Inputs:** APTOS 2019 Blindness Detection (competition); IDRiD (`aaryapatel98/indian-diabetic-retinopathy-image-dataset`).

`%%bash` must be the first line of its cell; put notes in markdown cells, not above it.

In [ ]:
import os, json
os.chdir('/kaggle/working')
if not os.path.exists('retina-screen'):
    !git clone -q https://github.com/ashwinruke/retina-screen.git
%cd /kaggle/working/retina-screen
!git pull -q
!pip install -q timm albumentations onnx onnxscript onnxruntime
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!nvidia-smi --query-gpu=name --format=csv,noheader

## Data, training and evaluation (~45 min)
Cut points are fit on APTOS validation data (every band ≥ 0.5 wide) and frozen; IDRiD is never used for tuning.

In [ ]:
%%bash
set -e
cd /kaggle/working/retina-screen
python scripts/prepare_data.py --aptos auto --idrid auto --out-dir data/manifests
python scripts/cache_images.py --manifest data/manifests/aptos.csv --out-dir cache --config configs/baseline_aptos.yaml
python scripts/cache_images.py --manifest data/manifests/idrid.csv --out-dir cache --config configs/baseline_aptos.yaml
rm -rf outputs/baseline_aptos outputs/eval
python scripts/train.py --config configs/baseline_aptos.yaml
python scripts/evaluate.py --checkpoint outputs/baseline_aptos/best.pt \
    --manifest cache/aptos.csv --splits test \
    --fit-manifest cache/aptos.csv --fit-splits val \
    --name mvp_aptos_test --export outputs/model.pt
python scripts/evaluate.py --checkpoint outputs/model.pt --manifest cache/idrid.csv --splits all  --name A0_idrid
python scripts/evaluate.py --checkpoint outputs/model.pt --manifest cache/idrid.csv --splits test --name A0_idrid_test
python scripts/plot_results.py --history outputs/baseline_aptos/history.csv \
    --report outputs/eval/mvp_aptos_test.json \
    --predictions outputs/eval/mvp_aptos_test_predictions.csv --out-dir docs/figures --prefix mvp
python scripts/plot_results.py --report outputs/eval/A0_idrid.json \
    --predictions outputs/eval/A0_idrid_predictions.csv --out-dir docs/figures --prefix idrid

## Save the results
Kaggle deletes `/kaggle/working` when the session ends.

In [ ]:
!python scripts/artifacts.py push --tag mvp --run baseline_aptos
!python scripts/artifacts.py list

In [ ]:
def show(*names):
    for name in names:
        r = json.load(open(f'outputs/eval/{name}.json'))['tuned_thresholds']
        rec = r['per_grade_recall']
        print(f"{name:20s} n={r['n']:4d} QWK={r['qwk']:.3f} AUC={r['referable']['auc']:.3f} "
              f"sens={r['referable']['sensitivity']:.3f} spec={r['referable']['specificity']:.3f} "
              f"severe={rec['Severe NPDR']:.2f} prolif={rec['Proliferative DR']:.2f} "
              f"cuts={[round(t, 2) for t in r['thresholds']]}")
show('mvp_aptos_test', 'A0_idrid', 'A0_idrid_test')

## Deploy the in-browser demo
The export checks the ONNX model against PyTorch and stops if they disagree.

In [ ]:
!python scripts/export_onnx.py --model outputs/model.pt --out-dir web \
    --check-images data/manifests/aptos.csv --report outputs/eval/mvp_aptos_test.json
!python scripts/deploy_space.py --space ashwinruke/retina-dr-screening